# AIC and BIC: Choosing the Right Model Without Overfitting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/statistics/aic_bic_model_selection.ipynb)

Derive the log-likelihood from first principles, then build AIC and BIC as penalised model selection criteria.

**Blog post:** [AIC and BIC: Choosing the Right Model Without Overfitting](https://sesen.ai/blog/aic-bic-model-selection-information-criteria)

**Key papers:**
- Akaike, H. (1974). A new look at the statistical model identification. *IEEE Trans. Automatic Control*, 19(6), 716-723.
- Schwarz, G. (1978). Estimating the dimension of a model. *The Annals of Statistics*, 6(2), 461-464.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)

## 1. Generate Synthetic Data

The true function is a cubic polynomial with Gaussian noise.

In [ ]:
n = 50
x = np.linspace(-3, 3, n)
y_true = 0.5 * x**3 - 2 * x + 1
y = y_true + np.random.normal(0, 3, n)

plt.figure(figsize=(8, 4))
plt.scatter(x, y, alpha=0.6, s=30, label='Observed data')
plt.plot(x, y_true, 'k--', alpha=0.4, label='True function')
plt.xlabel('x'); plt.ylabel('y')
plt.legend(); plt.grid(True, alpha=0.3)
plt.title('Synthetic Data: Cubic + Gaussian Noise')
plt.tight_layout()
plt.show()

## 2. Quick Win: The R Code Translation

The original R code manually computed the log-likelihood for linear models.

In [ ]:
# Fit a simple linear model
coeffs_1 = np.polyfit(x, y, 1)
y_pred_1 = np.polyval(coeffs_1, x)
residuals_1 = y - y_pred_1

# ML estimate of sigma (divide by n, not n-p)
sigma_ml_1 = np.sqrt(np.mean(residuals_1**2))

# Manual log-likelihood: sum of log-normal densities
ll_manual = np.sum(norm.logpdf(residuals_1, loc=0, scale=sigma_ml_1))
print(f"Manual log-likelihood (degree 1): {ll_manual:.4f}")
print(f"Sigma (ML):                       {sigma_ml_1:.4f}")

## 3. Compute AIC and BIC for Polynomial Models

In [ ]:
results = []
for degree in range(1, 11):
    coeffs = np.polyfit(x, y, degree)
    y_pred = np.polyval(coeffs, x)
    residuals = y - y_pred
    sigma_ml = np.sqrt(np.mean(residuals**2))
    ll = np.sum(norm.logpdf(y, loc=y_pred, scale=sigma_ml))
    k = degree + 2  # coefficients + sigma
    aic = 2 * k - 2 * ll
    bic = k * np.log(n) - 2 * ll
    results.append({'degree': degree, 'k': k, 'll': ll, 'aic': aic, 'bic': bic})
    print(f"Degree {degree:2d}: k={k:2d}, LL={ll:.2f}, AIC={aic:.2f}, BIC={bic:.2f}")

aic_vals = [r['aic'] for r in results]
bic_vals = [r['bic'] for r in results]
print(f"\nBest AIC: degree {results[np.argmin(aic_vals)]['degree']}")
print(f"Best BIC: degree {results[np.argmin(bic_vals)]['degree']}")

## 4. Plotting AIC and BIC vs Model Complexity

In [ ]:
degrees = [r['degree'] for r in results]
lls = [r['ll'] for r in results]

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(degrees, aic_vals, 'o-', color='#2196F3', linewidth=2, markersize=8, label='AIC')
ax1.plot(degrees, bic_vals, 's-', color='#FF9800', linewidth=2, markersize=8, label='BIC')

ax2 = ax1.twinx()
ax2.plot(degrees, lls, '^--', color='#4CAF50', linewidth=1.5, markersize=6, alpha=0.7, label='Log-Likelihood')
ax2.set_ylabel('Log-Likelihood', fontsize=12, color='#4CAF50')

ax1.set_xlabel('Polynomial Degree', fontsize=12)
ax1.set_ylabel('Information Criterion (lower is better)', fontsize=12)
ax1.set_title('AIC and BIC vs Model Complexity', fontsize=14)
ax1.set_xticks(degrees)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=10)
ax1.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 5. Underfitting vs Optimal vs Overfitting

In [ ]:
x_smooth = np.linspace(-3, 3, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
panels = [
    (1, 'Degree 1 (Underfitting)', '#E53935'),
    (3, 'Degree 3 (Optimal)', '#43A047'),
    (10, 'Degree 10 (Overfitting)', '#FF9800'),
]

for ax, (degree, title, color) in zip(axes, panels):
    coeffs = np.polyfit(x, y, degree)
    y_smooth = np.polyval(coeffs, x_smooth)
    y_pred = np.polyval(coeffs, x)
    residuals = y - y_pred
    sigma_ml = np.sqrt(np.mean(residuals**2))
    ll = np.sum(norm.logpdf(y, loc=y_pred, scale=sigma_ml))
    k = degree + 2
    aic = 2 * k - 2 * ll
    bic = k * np.log(n) - 2 * ll
    
    ax.scatter(x, y, c='#2196F3', alpha=0.6, s=25, edgecolors='white', linewidths=0.5)
    ax.plot(x_smooth, y_true_smooth := 0.5 * x_smooth**3 - 2 * x_smooth + 1, 'k--', alpha=0.3)
    ax.plot(x_smooth, y_smooth, color=color, linewidth=2.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(-12, 18); ax.set_xlim(-3.3, 3.3)
    ax.grid(True, alpha=0.3)
    ax.text(0.05, 0.95, f'AIC={aic:.1f}\nBIC={bic:.1f}', transform=ax.transAxes,
            fontsize=9, va='top', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

fig.suptitle('The Bias-Variance Tradeoff in Action', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 6. Cross-Validation Comparison

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []
for degree in range(1, 11):
    pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    scores = cross_val_score(pipe, x.reshape(-1, 1), y, cv=kf,
                             scoring='neg_mean_squared_error')
    cv_scores.append(-scores.mean())
    print(f"Degree {degree:2d}: CV MSE = {-scores.mean():.3f} ± {scores.std():.3f}")

print(f"\nBest CV: degree {np.argmin(cv_scores) + 1}")

In [ ]:
# Normalised comparison
aic_norm = (np.array(aic_vals) - min(aic_vals)) / (max(aic_vals) - min(aic_vals))
bic_norm = (np.array(bic_vals) - min(bic_vals)) / (max(bic_vals) - min(bic_vals))
cv_norm = (np.array(cv_scores) - min(cv_scores)) / (max(cv_scores) - min(cv_scores))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, 11), aic_norm, 'o-', color='#2196F3', linewidth=2, markersize=8, label='AIC')
ax.plot(range(1, 11), bic_norm, 's-', color='#FF9800', linewidth=2, markersize=8, label='BIC')
ax.plot(range(1, 11), cv_norm, 'D-', color='#9C27B0', linewidth=2, markersize=8, label='5-Fold CV MSE')
ax.axvline(x=3, color='green', linestyle='--', alpha=0.5, linewidth=1.5, label='True degree (3)')
ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('Normalised Score (lower is better)', fontsize=12)
ax.set_title('AIC vs BIC vs Cross-Validation', fontsize=14)
ax.set_xticks(range(1, 11))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Exercises

1. **Change the noise level.** Set the noise standard deviation to 1 instead of 3. Do AIC and BIC still agree?

2. **Implement AICc.** Add the small-sample correction `AICc = AIC + 2k(k+1)/(n-k-1)` and compare with standard AIC for n=20.

3. **Apply to real data.** Load the Boston housing dataset from scikit-learn, fit models with different feature subsets, and use AIC/BIC to choose the best.

4. **BIC for cluster selection.** Generate data from 3 Gaussian clusters, fit GMMs with 1-6 components, and use BIC to recover the correct number of clusters.